# Template to follow per NESP Dataset

| | |  
|---|---|  
| Summary (a TL;DR) | | 
| Author:| Thomas Galindo |
| Edited:| 2025-02-02 |
| Affiliation:| AODN/IMOS |
| e-mail:| thomas.galindo@utas.edu.au |
| Date of creation:| 2025-02-02 |
| Date of last update:| 2025-02-02 |

## Background

This is a template notebook outlining the library stack and content structure for NESP Data Products.

It outlines:
1. [Connecting to NESP 5.9 datasets](#connecting-to-nesp-59-datasets)
2. [H3 Aggregation and Mapping](#h3-aggregation-and-mapping)
3. [Identifying Trend Sites](#identifying-trend-sites)

## NESP 5.9 Datasets

| Dataset name | Description | Metadata | S3 URL |
| ------------ | ----------- | -------- | ------ |
| AMSA         | TODO        | TODO     | [link](s3://data-uplift-public/stored/datauplift/amsa/)                                   |
| Kelp         | TODO        | TODO     | [link](s3://data-uplift-public/stored/datauplift/kelp/kelp.parquet)                       |
| NRMN         | TODO        | TODO     | [link](s3://data-uplift-public/stored/datauplift/nrmn/nrmn.parquet)    |
| Seabird      | TODO        | TODO     | [link](s3://data-uplift-public/stored/datauplift/seabird/seabird.parquet)                 |
| Seagrass     | TODO        | TODO     | [link](s3://data-uplift-public/stored/datauplift/seagrass/seagrass.parquet)               |

### S3 Details
NESP 5.9 datasets are currently stored in S3 and are publicly available as per the following table:

| Bucket               | Key                                                      | Partitioned |
| -------------------- | -------------------------------------------------------- | ----------- |
| `data-uplift-public` | `stored/datauplift/amsa/year=*/source=*/*.parquet`       | [x]         |
| `data-uplift-public` | `stored/datauplift/kelp/kelp.parquet`                    | [ ]         |
| `data-uplift-public` | `stored/datauplift/nrmn/nrmn.parquet` | [ ]         |
| `data-uplift-public` | `stored/datauplift/seabird/seabird.parquet`              | [ ]         |
| `data-uplift-public` | `stored/datauplift/seagrass/seagrass.parquet`            | [ ]         |

## Required Python & Packages
The required packages are found in the [pyproject.toml](pyproject.toml) under `dependencies`.

### `uv` installation
It is highly recommended to manage python version, environment and projects with [uv](https://github.com/astral-sh/uv).

```bash
uv venv && source .venv/bin/activate && uv pip install .
```

### `pip` installation
Otherwise, native python tooling can be used to manage the python environment and project. 

*Note that python3.12 or greater is required.*

```bash
python -m venv .venv && source .venv/bin/activate && pip install .
```

### Running Notebooks in Jupyter or IDE
The notebook can be manually run by launching jupyter notebook from the console:
```bash
jupyter notebook
```
Otherwise, if running in an IDE, select the newly created `.venv` as the kernal.


## Notebook Stack

### IO
`pyarrow` is the defacto parquet and arrow standard library tooling in the python eco-system, offering exceptional performance, documentation and stability.

Its `fs` class allows connecting to parquet datasets in AWS S3, which is perfect for our use case.

### Computation
`polars` is fast replacing pandas as the primary python dataframe API. It is more tightly integrated with arrow (and pyarrow), has exceptional performance, greater than memory dataset manipulation, a more consistent and declarative API, built in plotting and many other advantages.

### Mapping
`pydeck` is a python binding for `deck.gl`, a GPU-powered framework for visual exploratory data analysis of large datasets. It is optimised for Jupyter notebooks and highly customisable.

Standard mapping tools like `folium` struggle to handle the number of mapped shapes, so we favor the more optimised `pydeck` library.

### Rich
TODO: Update on `rich` library

In [ ]:
# Set up rich console
import rich
import rich.console
console = rich.console.Console()

## Connecting to NESP 5.9 Datasets

NESP 5.9 datasets can be connected to via a pyarrow `S3FileSystem`.

The `S3FileSystem` manages the connection to the NESP 5.9 datasets stored in S3, resulting in a `pyarrow.Dataset` handle that we may use for efficient loading and subsetting of the dataset.

Note that a dataset can be comprised of a single parquet file or partitioned (into separate parquet fragments/files).

This step also makes the dataset schema available and file structure available allowing for pre-filtering without downloading the entire dataset.

Finally, we may estimate the in memory dataset size by sampling the dataset. If the dataset is small, we may use a `DataFrame` straight away. Otherwise if the dataset is too large, we must pre-aggregate and or filter using a `LazyFrame` before loading the dataset into memory.

In [ ]:
import pyarrow
import pyarrow.fs
import pyarrow.dataset
import util

# Construct the anonymous file system responsible for reading from the public S3 bucket
FILE_SYSTEM = pyarrow.fs.S3FileSystem(
    region="ap-southeast-2", 
    anonymous=True,
)

# Create the dataset connection
# By convention, datasets are labelled `ds`
ds = pyarrow.dataset.dataset(
    source="data-uplift-public/stored/datauplift/seabird/seabird.parquet",
    filesystem=FILE_SYSTEM,
)

# The comprising files of the dataset can be inspected
rich.print(list(ds.get_fragments()))

# The schema of the dataset can be inspected
console.print(
    util.generate_schema_rich_table(
        schema=ds.schema,
        metadata_keys=["definition", "units", "long_name", "resolution"]
    )
)

# For the full table scheme use native pyarrow representation
# rich.print(ds.schema)

# Estimate dataset size
estimated_megabytes = util.estimate_dataset_size(
    ds=ds,
    n_samples=10_000,
)
rich.print(f"DS estimated size: {round(estimated_megabytes, 2)}MB")

## Loading Data
The dataset handle is our connection to data in S3. We can use this handle to load data directly into a `DataFrame` or `LazyFrame`

### DataFrame
For smaller datasets the `DataFrame` is preferrable. This loads the entire dataset into memory ready for analysis.

### LazyFrame
For larger datasets the `DataFrame` may not fit into memory. In these cases the `LazyFrame` is ideal as it allows the dataset to be filtered and or aggregated prior to loading the dataset, allowing it to fit into memory, ready for analysis.

In [ ]:
import polars
import polars_h3

# Option (a): Load entire dataset in memory as a dataframe
# Suitable for smaller datasets that fit into memory
df = polars.DataFrame(
    data=ds.to_table(),
)

# Option (b): Scan entire dataset as a lazyframe
# Suitable for larger datasets that don't fit into memory
lf = polars.scan_pyarrow_dataset(
    source=ds,
)

## H3 Aggregation and Mapping
The NESP 5.9 datasets are aggregated data products that cover large regions and time periods.

It is useful to first aggregate the datasets so that we may understand their temporal-spatial bounds.

### H3 Indexing
The `h3` library allows us to spatially index datasets at different [resolutions](https://h3geo.org/docs/core-library/restable#average-area-in-km2) (zoom levels) according to the record latitude and longitude.

Additionally, the `polars_h3` library offers a polars style interface for faster tagging with `polars.DataFrames`:
```python

# Add a h3 index to any dataframe with latitude and longitude columns
df = df.with_columns(

    # Polars Expression Shorthand to access the `polars_h3` namespace
    polars_h3.latlng_to_cell(

        # Specify the latitude and longitude columns
        lat=polars.col("decimalLatitude"),
        lng=polars.col("decimalLongitude"),

        # Specify the resolution
        resolution=5,

        # Specify the index return type, can be `polars.Int64` or `polars.String`
        return_dtype=polars.String,
    )
    
    # Alias to `h3Index`
    .alias("h3Index"),
)
```

### H3 Mapping
The `pydeck` library offers a built in [`H3HexagonLayer`](https://deck.gl/docs/api-reference/geo-layers/h3-hexagon-layer) layer type that greatly reduces complexity in mapping H3 Shapes:

```python

# Generate a h3 hexagon layer displayable on a pydeck map
pydeck.Layer(

    # Specify this is a H3 Hexagon Layer
    "H3HexagonLayer",

    # Unfortunately no polars dataframe support, so cast to `pandas.DataFrame`
    df.to_pandas(use_pyarrow_extension_array=True),

    # Specify the hexagon
    get_hexagon="h3Index",

)
```
This is rendered with a [`pydeck.Deck`](https://deck.gl/docs/api-reference/core/deck):
```python

# Generate a deck with h3 layers
deck = pydeck.Deck(

    # Supply the h3 layers
    layers=h3_layers,
)
deck.show()
```

In [ ]:
import pydeck

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(

        # H3 Aggregated dataframe with dataset and number of records attributes
        df
        .group_by(
            polars.col("h3Index"),
        )
        .agg(
            polars.col("id").count().alias("n_records"),
            polars.col("datasetName"),
        )
        .with_columns(
            polars.col("datasetName").list.unique().list.sort().list.join(", ").alias("datasets"),
        )
    ),
    aggregate_column_name="n_records",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=util.GLOBAL_VIEW_STATE,
)
deck.show()

## Australian Marine Regions
Each occurrence has been tagged by the following Australian Marine Regions (see table below). Note that the taggin is not 100% accurate at the region boundaries; it is accurate to ~6km.

Note that the Great Barrier Reef has been included from the World Heritage Areas map.

| Marine Region Type | Map Explore URL | Metadata URL (REST/About) | GeoJSON API Endpoint |
| :--- | :--- | :--- | :--- |
| **Commonwealth Marine Regions** | [Explore](https://fed.dcceew.gov.au/datasets/commonwealth-marine-regions/explore) | [Metadata](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/Commonwealth_Marine_Regions/MapServer/0) | [GeoJSON](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/Commonwealth_Marine_Regions/MapServer/0/query?outFields=*&where=1%3D1&f=geojson) |
| **World Heritage Areas** | [Explore](https://fed.dcceew.gov.au/datasets/erin::australia-world-heritage-areas/explore) | [Metadata](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/World_Heritage_Areas/MapServer/0) | [GeoJSON](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/World_Heritage_Areas/MapServer/0/query?outFields=*&where=1%3D1&f=geojson) |
| **Marine Parks** | [Explore](https://fed.dcceew.gov.au/datasets/2b3eb1d42b8d4319900cf4777f0a83b9_0/explore) | [Metadata](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/Australian_Marine_Parks/MapServer/0) | [GeoJSON](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/Australian_Marine_Parks/MapServer/0/query?outFields=*&where=1%3D1&f=geojson) |
| **IMCRA Provincial** | [Explore](https://fed.dcceew.gov.au/datasets/erin::integrated-marine-and-coastal-regionalisation-of-australia-imcra-v4-0-provincial-bioregions/explore) | [Metadata](https://fed.dcceew.gov.au/datasets/erin::integrated-marine-and-coastal-regionalisation-of-australia-imcra-v4-0-provincial-bioregions/about) | [GeoJSON](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/IMCRA_Provincial/MapServer/0/query?outFields=*&where=1%3D1&f=geojson) |
| **IMCRA Mesoscale** | [Explore](https://fed.dcceew.gov.au/datasets/erin::integrated-marine-and-coastal-regionalisation-of-australia-imcra-v4-0-meso-scale-bioregions/explore) | [Metadata](https://fed.dcceew.gov.au/datasets/erin::integrated-marine-and-coastal-regionalisation-of-australia-imcra-v4-0-meso-scale-bioregions/about) | [GeoJSON](https://gis.environment.gov.au/gispubmap/rest/services/ogc_services/IMCRA_Mesoscale/MapServer/0/query?outFields=*&where=1%3D1&f=geojson) |

Follow the **Explore** links to explore the Australian Marine Regions. The Commonwealth Marine Regions explore map has been embedded below.

In [ ]:
import IPython.display

# Display the Commonwealth Marine Regions
IPython.display.IFrame(
    src="https://fed.dcceew.gov.au/datasets/commonwealth-marine-regions/explore?location=-33.665700%2C140.517300%2C4",
    width="100%", 
    height="1000px",
)

### Exploring Available Marine Regions Tags

Australian Marine Regions Tags have the following syntax:

| TAG | TYPE | OPTIONAL SUPER REGIONS | REGION |
| :--- | :--- | :--- | :--- |
| `Commonwealth Marine Region: South-east` | `Commonwealth Marine Region` | | `South-east` |
| `Marine Park:National Park Zone:Freycinet` | `Marine Park` | `National Park Zone` | `Freycinet` |
| `IMCRA:Provincial:Water Type:Cold Temperate Waters` | `IMCRA` | `Provincial`, `Water Type` | `Cold Temperate Waters` |

Extracting the available Australian Marine Regions Tags requires spliting the delimited column:

In [ ]:
# Extract the unique tags
australian_marine_regions_tags = df["australianMarineRegionsTags"].str.split("|").explode().value_counts(sort=True).drop_nulls()

# Maniuplate polars display config to display full tag names
with polars.Config(
    fmt_str_lengths=128,
    tbl_rows=20
):
    display(australian_marine_regions_tags)

### Filtering to All Data within Australian Marine Regions
We may filter to a particular requires a using not `is_null` constraint:

In [ ]:
# Filter to the commonwealth marine regions
australian_marine_regions_df = df.filter(
    polars.col("australianMarineRegionsTags").is_null().not_()
)

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(
        australian_marine_regions_df

        # Additional re-index at a higher resolution
        .with_columns(
            polars_h3.latlng_to_cell(
                lat=polars.col("decimalLatitude"),
                lng=polars.col("decimalLongitude"),
                resolution=5,
                return_dtype=polars.String,
            ).alias("h3Index"),
        )
        .group_by(
            polars.col("h3Index"),
        )
        .agg(
            polars.col("id").count().alias("n_records"),
            polars.col("datasetName"),
        )
        .with_columns(
            polars.col("datasetName").list.unique().list.sort().list.join(", ").alias("datasets"),
        )
    ),
    aggregate_column_name="n_records",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=util.generate_view_state(df=australian_marine_regions_df)
)
deck.show()

### Filtering to a particular Marine Region
We may filter to a particular requires a using `str.contains` expression:

In [ ]:
# Filter to the commonwealth marine regions
australian_marine_regions_df = df.filter(
    polars.col("australianMarineRegionsTags").str.contains(
        pattern="Commonwealth Marine Region:South-east",
        literal=True,
    )
)

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(
        australian_marine_regions_df

        # Additional re-index at a higher resolution
        .with_columns(
            polars_h3.latlng_to_cell(
                lat=polars.col("decimalLatitude"),
                lng=polars.col("decimalLongitude"),
                resolution=5,
                return_dtype=polars.String,
            ).alias("h3Index"),
        )
        .group_by(
            polars.col("h3Index"),
        )
        .agg(
            polars.col("id").count().alias("n_records"),
            polars.col("datasetName"),
        )
        .with_columns(
            polars.col("datasetName").list.unique().list.sort().list.join(", ").alias("datasets"),
        )
    ),
    aggregate_column_name="n_records",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=util.generate_view_state(df=australian_marine_regions_df)
)
deck.show()

## Identifying Trend Sites
In the global map we can see there are some sites with a high density of records. These are the most ideal sites for trend analysis.

### Per Region
[Regions](#australian-marine-regions) with a high density of records are typically good candidates for trend analysis.

### Per Dataset
Additionally, the singular datasets that comprise the aggregated data product are typically good candidates for trend analysis.

### Observation Effort vs Organism Quantity
Observation effort is the number of occurrence records while Organism Quantity is the recorded count for a specific species.

Observation effort must be taken into account when analysing trends; fluctuations in observation effort will distort the reported occurrences.

In [ ]:
yearly_observation_intensity = (
    df
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("datasetName"),
    )
    .agg(
        polars.col("id").count().alias("nRecord"),
    )
    .plot.bar(
        x="year",
        y="nRecord",
        color="datasetName",
    )
)
yearly_quantity_intensity = (
    df
    # Some organismQuantity values are "present", which must be filtered out
    .filter(
        polars.col("organismQuantity").cast(polars.Int64, strict=False).is_null().not_(),
    )
    .with_columns(
        polars.col("organismQuantity").cast(polars.Int64),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("datasetName"),
    )
    .agg(
        polars.col("organismQuantity").sum(),
    )
    .plot.bar(
        x="year",
        y="organismQuantity",
        color="datasetName",
    )
)
yearly_observation_intensity & yearly_quantity_intensity

### Mapping the Spatial Extent of a Particular Dataset
Similarly to mapping Australian Marine Regions, we may map particular datasets by applying a string constraint:

In [ ]:
# Filter to the `seabird atlas sea` dataset
seabird_atlas_sea_df = df.filter(
    polars.col("datasetName").eq("seabird_atlas_southeast_australia"),
)

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(
        seabird_atlas_sea_df

        # Additional re-index at a higher resolution
        .with_columns(
            polars_h3.latlng_to_cell(
                lat=polars.col("decimalLatitude"),
                lng=polars.col("decimalLongitude"),
                resolution=6,
                return_dtype=polars.String,
            ).alias("h3Index"),
        )
        .group_by(
            polars.col("h3Index"),
        )
        .agg(
            polars.col("id").count().alias("n_records"),
            polars.col("datasetName"),
        )
        .with_columns(
            polars.col("datasetName").list.unique().list.sort().list.join(", ").alias("datasets"),
        )
    ),
    aggregate_column_name="n_records",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=util.generate_view_state(df=seabird_atlas_sea_df)
)
deck.show()

### Identifying the Most Occurring Species

In [ ]:
top_ten_species_df = seabird_atlas_sea_df["scientificName"].value_counts(sort=True).head(10)
top_ten_species_list = top_ten_species_df["scientificName"].to_list()
display(top_ten_species_df)

### Plotting the Species Occurrences Over Time (Yearly)

In [ ]:
# Observe the large fluctuations in observation effort in even a single dataset
# This requires normalisation in future graphing
(
    seabird_atlas_sea_df
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
    )
    .agg(
        polars.len().alias("observation_effort"),
    )
    .plot.line(
        x="year",
        y="observation_effort",
    )
)

In [ ]:
# Observe some species quantity (rarity/observation focus) varies wildy as well
# This requires normalisation in future graphing
(
    seabird_atlas_sea_df
    .filter(
        polars.col("scientificName").is_in(
            seabird_atlas_sea_df["scientificName"].value_counts(sort=True)["scientificName"][:21].to_list(),
        ),
        polars.col("organismQuantity").cast(polars.Int64, strict=False).is_not_null(),
    )
    .group_by(
        polars.col("scientificName"),
        polars.col("datasetName"),
        polars.col("eventDate").dt.year().alias("year"),
    ).agg(
        polars.col("organismQuantity").cast(polars.Int64).sum(),
    ).sort(
        polars.col("organismQuantity"),
        descending=True,
    ).plot.bar(
        x="organismQuantity",
        y="scientificName",
        color="year",
    )
)

In [ ]:
# Produce a observation effort normalised chart utilising the top most occurring species
year_chart_normalized = (
    seabird_atlas_sea_df
    .filter(
        polars.col("scientificName").is_in(top_ten_species_list[:9]),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("scientificName"),
    )
    .agg(
        polars.col("organismQuantity").cast(polars.Int64).sum(),
        polars.col("id").count().alias("count"),
    )
    .with_columns(
        # This creates a column of total observation effort across for that year
        polars.col("count").sum().over("year").over("scientificName").alias("total_annual_occurrences"),
        polars.col("organismQuantity").sum().over("scientificName").alias("total_annual_quantities")
    )
    .with_columns(
        (polars.col("organismQuantity") / polars.col("total_annual_quantities")).clip(0).alias("organismQuantityNormalised"),
    )
    .plot.line(
        x="year",
        y="organismQuantityNormalised",
        color="scientificName",
    ).facet(
        facet="scientificName",
        columns=3,
    )
    .resolve_scale(
        y="independent",
    )
)
year_chart_normalized

In [ ]:
# Produce a combined effort and species-normalised chart
year_chart_combined = (
    seabird_atlas_sea_df
    .filter(
        polars.col("surveyType").ne("Tracking"),
    )
    .filter(
        polars.col("scientificName").is_in(
            seabird_atlas_sea_df["scientificName"].value_counts(sort=True)["scientificName"][:21].to_list(),
        ),
        polars.col("organismQuantity").cast(polars.Int64, strict=False).is_null().not_(),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("scientificName"),
        polars.col("datasetName"),
    )
    .agg(
        polars.col("organismQuantity").cast(polars.Int64).sum().alias("annual_sum"),
        polars.len().alias("species_records"),
    )
    .with_columns(
        # 1. Effort Normalisation: Total records (effort) across ALL species for that year
        polars.col("species_records").sum().over("year").alias("total_annual_effort"),
    )
    .with_columns(
        # Abundance per unit of effort
        (polars.col("annual_sum") / polars.col("total_annual_effort")).alias("abundance_per_effort"),
    )
    .with_columns(
        # 2. Species Normalisation: Mean abundance_per_effort for this species across all years
        polars.col("abundance_per_effort").mean().over("scientificName").alias("species_historical_effort_mean"),
    )
    .with_columns(
        # Final Index: How this year's effort-adjusted abundance compares to the species average
        (polars.col("abundance_per_effort") / polars.col("species_historical_effort_mean")).clip(0).alias("relative_abundance_index"),
    )
    .plot.line(
        x="year",
        y="relative_abundance_index",
        color="scientificName",
    )
    .facet(
        facet="scientificName",
        columns=3,
    )
    .resolve_scale(
        y="independent",
    )
)

year_chart_combined

In [ ]:
# Produce a combined effort and species-normalised chart
year_chart_combined = (
    seabird_atlas_sea_df
    .filter(
        polars.col("surveyType").ne("Tracking"),
    )
    .filter(
        polars.col("scientificNameID").is_in(
            seabird_atlas_sea_df["scientificNameID"].value_counts(sort=True)["scientificNameID"][:21].to_list(),
        ),
        polars.col("organismQuantity").cast(polars.Int64, strict=False).is_null().not_(),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("scientificNameID"),
        polars.col("datasetName"),
    )
    .agg(
        polars.col("organismQuantity").cast(polars.Int64).sum().alias("annual_sum"),
        polars.len().alias("species_records"),
    )
    .with_columns(
        # 1. Effort Normalisation: Total records (effort) across ALL species for that year
        polars.col("species_records").sum().over("year").alias("total_annual_effort"),
    )
    .with_columns(
        # Abundance per unit of effort
        (polars.col("annual_sum") / polars.col("total_annual_effort")).alias("abundance_per_effort"),
    )
    .with_columns(
        # 2. Species Normalisation: Mean abundance_per_effort for this species across all years
        polars.col("abundance_per_effort").mean().over("scientificNameID").alias("species_historical_effort_mean"),
    )
    .with_columns(
        # Final Index: How this year's effort-adjusted abundance compares to the species average
        (polars.col("abundance_per_effort") / polars.col("species_historical_effort_mean")).clip(0).alias("relative_abundance_index"),
    )
    .plot.line(
        x="year",
        y="relative_abundance_index",
        color="scientificNameID",
    )
    .facet(
        facet="scientificNameID",
        columns=3,
    )
    .resolve_scale(
        y="independent",
    )
)

year_chart_combined